This notebook demonstrates an example of using MedSynth as a data augmentation strategy for the ACI-Bench dataset in the Dialogue-to-Note summarization task.

The idea of this notebook is to provide a self-contained example. All the code is from the MedSynth repository, just adjusted and edited to run as a **Google colab notebook**.

Note that the results reported in the paper are for `num_train_epochs = 4`. In this notebook we set `num_train_epochs=1` for time and compute limitations.

# Classes, functions, and constants

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
! pip install -q evaluate

In [5]:
! pip install -q rouge_score

In [1]:
import torch
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
base_model_path= "unsloth/llama-3-8b-Instruct"
summarizer_system_prompt= """You are an assistant for medical professionals, specializing in summarizing their conversations with patients. Your role is to accurately and comprehensively summarize these conversations in the SOAP (Subjective, Objective, Assessment, Plan) format. Ensure that each summary is thorough and precise, reflecting all relevant details from the conversation to provide a reliable medical record."""
model_evaluator_generation_config = {"max_new_tokens":3000,
                     "do_sample":True,
                     "temperature":0.6, #0.6
                     "top_p":0.9,
                     "use_cache": True,
                    }

tuning_config = {
    "hugging_face_username":"Ahmad0067",
    "model_config": {
        "base_model":"{BASE_MODEL}", # The base model
        "finetuned_model":"{FINE_TUNED_MODEL_NAME}", #"llama-3-8b-Instruct-aci-train", # The finetuned model
        "max_seq_length": 8192, # The maximum sequence length that the base model can handle.
        #"dtype":torch.bfloat16 , # The data type: changed from float16
        "load_in_4bit": True, # Load the model in 4-bit
    },
    "lora_config": {
      "r": 16, # The number of LoRA layers 8, 16, 32, 64
      "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # The target modules
      "lora_alpha":16, # The alpha value for LoRA
      "lora_dropout":0, # The dropout value for LoRA
      "bias":"none", # The bias for LoRA
      "use_gradient_checkpointing":True, # Use gradient checkpointing
      "use_rslora":False, # Use RSLora
      "use_dora":False, # Use DoRa
      "loftq_config":None # The LoFTQ configuration
    },
    "training_dataset":{
        "name":"{TRAINING_DATA_PATH_HF}",
        "split":"train", # The dataset split
        "input_field":"text", # The input field
    },
    "training_config": {
        "per_device_train_batch_size": 2, # The batch size
        "gradient_accumulation_steps": 4, # The gradient accumulation steps
        "warmup_steps": 5, # The warmup steps
        "max_steps":0, # The maximum steps (0 if the epochs are defined)
        "num_train_epochs": 1, # The number of training epochs(0 if the maximum steps are defined)
        "learning_rate": 2e-4, # The learning rate
        "fp16": not torch.cuda.is_bf16_supported(), # The fp16
        #"bf16": torch.cuda.is_bf16_supported(), # The bf16
        "logging_steps": 1, # The logging steps
        "optim" :"adamw_8bit", # The optimizer
        "weight_decay" : 0.01,  # The weight decay
        "lr_scheduler_type": "linear", # The learning rate scheduler
        "seed" : 42, # The seed
        "output_dir" : "outputs", # The output directory
    }
}

In [3]:
def get_model_responses(model, tokenizer, summarizer_system_promt, test_dataset, generation_config):
        dial_summary_pairs = {}
        for idx, conversation in enumerate (test_dataset["dialogue"]):
            print(f"processing idx: {idx}")

            inputs = tokenizer(
            [f"<|begin_of_text|><|start_header_id|>system<|end_header_id|> \n\n {{{{ {summarizer_system_promt} }}}}<|eot_id|><|start_header_id|>user<|end_header_id|> \n\n {{{{ This is the conversation: {conversation} }}}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"], return_tensors = "pt").to("cuda")

            outputs= model.generate(**inputs,
                                         max_new_tokens= generation_config["max_new_tokens"],
                                         use_cache = generation_config["use_cache"],
                                         do_sample= generation_config["do_sample"],
                                         temperature= generation_config["temperature"],
                                         top_p= generation_config["top_p"],)

            response= tokenizer.batch_decode(outputs, skip_special_tokens = False) #True
            start_index = response[0].rfind("<|start_header_id|>assistant<|end_header_id|>")+45
            #end_index = response[0].rfind("<|eot_id|>")

            # Extract the summary part
            summary = response[0][start_index:].strip()

            dial_summary_pairs[idx]= {"conversation": conversation, "summary": summary}

        return dial_summary_pairs

In [4]:
import evaluate
import numpy as np


class MetricsComputer:
    def __init__(self, prediction_list, gt_list):
        self.pred_list = prediction_list
        self.gt_list = gt_list

    def compute_BLEU(self):
        bleu = evaluate.load("bleu")
        results = bleu.compute(predictions=self.pred_list, references=self.gt_list)
        return results["bleu"]

    def compute_ROUGE(self):
        rouge = evaluate.load('rouge')
        results = rouge.compute(predictions=self.pred_list, references=self.gt_list)
        return results


    def compute_METEOR(self):
        meteor= evaluate.load('meteor')
        results = meteor.compute(predictions=self.pred_list, references=self.gt_list)
        return results["meteor"]




def get_automatic_eval_scores(dial_summary_pairs, test_dataset):
        dial_summary_pairs_df = pd.DataFrame.from_dict(dial_summary_pairs, orient='index')
        metrics_computer = MetricsComputer(prediction_list= dial_summary_pairs_df["summary"],
                                            gt_list= test_dataset["note"])


        return {
            'BLEU': metrics_computer.compute_BLEU(),
            'ROUGE-1': metrics_computer.compute_ROUGE()['rouge1'],
            'ROUGE-2': metrics_computer.compute_ROUGE()['rouge2'],
            'ROUGE-L': metrics_computer.compute_ROUGE()['rougeL'],
            'ROUGE-LSum': metrics_computer.compute_ROUGE()['rougeLsum'],
            "METEOR": metrics_computer.compute_METEOR(),
            }



In [5]:
# source: https://mlops.community/budget-instruction-fine-tuning-of-llama-3-8b-instructon-medical-data-with-hugging-face-google-colab-and-unsloth/
# source github (more update): https://github.com/Shekswess/LLM-Medical-Finetuning/blob/main/src/data_processing/create_process_datasets.py
from abc import ABC, abstractmethod

import pandas as pd



class InstructDataset(ABC):
    """
    Abstract class for creating Instruct Datasets
    """

    def __init__(self, dataset_path: str):
        """
        Initialize the dataset
        :param dataset_path: The path to the dataset
        """
        self.dataset = None
        self.load_dataset(dataset_path)

    def load_dataset(self, dataset_path: str) -> None:
        """
        Load the dataset from the given path
        :param dataset_path: The path to the dataset
        :return: None
        """
        self.dataset = pd.read_csv(dataset_path)

    def rename_columns(self, columns: dict[str, str]) -> None:
        """
        Rename the columns of the dataset
        :param columns: A dictionary of the form {old_name: new_name}
        :return: None
        """
        self.dataset = self.dataset.rename(columns=columns)

    def drop_columns(self, columns: list[str]) -> None:
        """
        Drop the columns from the dataset
        :param columns: A list of column names to drop
        :return: None
        """
        drop_columns = [col for col in columns if col in self.dataset.columns]
        self.dataset = self.dataset.drop(columns=drop_columns)

    def drop_bad_rows(self, columns: list[str]) -> None:
        """
        Drop the rows which have bad values in the columns
        :param columns: A list of columns to check for bad values
        :return: None
        """
        self.dataset = self.dataset.dropna(subset=columns)
        self.dataset = self.dataset.drop_duplicates(subset=columns)

    def create_instruction(self, instruction: str) -> None:
        """
        Create an instruction column in the dataset
        :param instruction: The instruction to add to the dataset
        :return: None
        """
        self.dataset["instruction"] = instruction

    @abstractmethod
    def create_prompt(self) -> None:
        """
        Create the prompt column in the dataset
        :return: None
        """
        pass

    def get_dataset(self) -> pd.DataFrame:
        """
        Get the dataset
        :return: The dataset
        """
        return self.dataset




class Llama3InstructDataset(InstructDataset):
    # source for edits: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/discussions/14

    def create_prompt(self):
        """
        Create the prompt column in the dataset which will be used for
        """
        prompts = []
        for index, row in self.dataset.iterrows():
            prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|> \n\n {{{{ {row['instruction']} }}}}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n  {{{{ This is the conversation: {row['Dialogue']} }}}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n {{{{ {row['Note']} }}}}<|eot_id|>"""
            prompts.append(prompt)
            if index==0:
                print(prompt)
        self.dataset["prompt"] = prompts


In [6]:
from datasets import Dataset, DatasetDict
import os


def process_dataset(dataset_path: str, model: str) -> pd.DataFrame:
    """
    Process the instruct dataset to be in the format required by the model.
    :param dataset_path: The path to the dataset.
    :param model: The model to process the dataset for.
    :return: The processed dataset.
    """
    if model == "gemma":
        pass
    elif model == "mistral":
        pass
    elif model == "llama":
        pass
    elif model == "llama3":
        dataset = Llama3InstructDataset(dataset_path)
    else:
        raise ValueError(f"Model {model} not supported!")

    dataset.create_instruction(summarizer_system_prompt)
    dataset.create_prompt()

    return dataset.get_dataset()



def create_dataset_hf(
    dataset: pd.DataFrame,
) -> DatasetDict:
    """
    Create a Hugging Face dataset from the pandas dataframe.
    :param dataset: The pandas dataframe.
    :return: The Hugging Face dataset.
    """
    dataset.reset_index(drop=True, inplace=True)
    return DatasetDict({"train": Dataset.from_pandas(dataset)})


In [7]:
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
from datasets import load_dataset


class ModelTuner:

    def __init__(self, TRAINING_DATA_PATH_HF, FINE_TUNED_MODEL_NAME, tuning_config, base_model):

        self.tuning_config = tuning_config
        # Update the tuning configuration with actual values provided
        self.tuning_config['model_config']['base_model'] = base_model
        self.tuning_config['model_config']['finetuned_model'] = FINE_TUNED_MODEL_NAME
        self.tuning_config['training_dataset']['name'] = TRAINING_DATA_PATH_HF


    def _load_model_and_tokenizer(self):
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name = self.tuning_config.get("model_config").get("base_model"),
            max_seq_length = self.tuning_config.get("model_config").get("max_seq_length"),
            dtype = self.tuning_config.get("model_config").get("dtype"),
            load_in_4bit = self.tuning_config.get("model_config").get("load_in_4bit"),
        )



    def _prepare_model_for_peft(self):
        # Setup for QLoRA/LoRA peft of the base model
        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r = self.tuning_config.get("lora_config").get("r"),
            target_modules = self.tuning_config.get("lora_config").get("target_modules"),
            lora_alpha = self.tuning_config.get("lora_config").get("lora_alpha"),
            lora_dropout = self.tuning_config.get("lora_config").get("lora_dropout"),
            bias = self.tuning_config.get("lora_config").get("bias"),
            use_gradient_checkpointing = self.tuning_config.get("lora_config").get("use_gradient_checkpointing"),
            random_state = 42,
            use_rslora = self.tuning_config.get("lora_config").get("use_rslora"),
            use_dora = self.tuning_config.get("lora_config").get("use_dora"),
            loftq_config = self.tuning_config.get("lora_config").get("loftq_config"),
        )



    def _load_training_data(self):
        # Loading the training dataset
        self.dataset_train = load_dataset(self.tuning_config.get("training_dataset").get("name"), split = self.tuning_config.get("training_dataset").get("split"))


    def _prepare_trainer(self):
        self.trainer = SFTTrainer(
            model = self.model,
            tokenizer = self.tokenizer,
            train_dataset = self.dataset_train,
            dataset_text_field = self.tuning_config.get("training_dataset").get("input_field"),
            max_seq_length = self.tuning_config.get("model_config").get("max_seq_length"),
            dataset_num_proc = 2,
            packing = False,
            args = TrainingArguments(
                per_device_train_batch_size = self.tuning_config.get("training_config").get("per_device_train_batch_size"),
                gradient_accumulation_steps = self.tuning_config.get("training_config").get("gradient_accumulation_steps"),
                warmup_steps = self.tuning_config.get("training_config").get("warmup_steps"),
                max_steps = self.tuning_config.get("training_config").get("max_steps"),
                num_train_epochs= self.tuning_config.get("training_config").get("num_train_epochs"),
                learning_rate = self.tuning_config.get("training_config").get("learning_rate"),
                fp16 = self.tuning_config.get("training_config").get("fp16"),
                #bf16 = self.tuning_config.get("training_config").get("bf16"),
                logging_steps = self.tuning_config.get("training_config").get("logging_steps"),
                optim = self.tuning_config.get("training_config").get("optim"),
                weight_decay = self.tuning_config.get("training_config").get("weight_decay"),
                lr_scheduler_type = self.tuning_config.get("training_config").get("lr_scheduler_type"),
                seed = 42,
                output_dir = self.tuning_config.get("training_config").get("output_dir"),
                ),

        )



    def model_train_and_save(self):
        self._load_model_and_tokenizer()
        self._prepare_model_for_peft()
        self._load_training_data()
        self._prepare_trainer()

        self.trainer.train()
        # saving the model to the hub:
        self.model.push_to_hub(self.tuning_config.get("model_config").get("finetuned_model"), tokenizer= self.tokenizer)


# Loading Aci-Bench Test set:

In [8]:
import pandas as pd


In [9]:
aci_test= pd.read_csv("/content/clinicalnlp_taskC_test2.csv")
aci_test.head()

,dataset,encounter_id,dialogue,note
0,virtassist,D2N128,"[doctor] hi , carolyn . how are you ?\n[patien...",CHIEF COMPLAINT\n\nFollow-up of chronic proble...
1,virtassist,D2N129,"[doctor] good afternoon , beverly . good to se...",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...
2,virtassist,D2N130,"[doctor] hi , anna , how are you ?\n[patient] ...",CHIEF COMPLAINT\n\nJoint pain.\n\nHISTORY OF P...
3,virtassist,D2N131,"hi , susan , how are you ?\n[patient] good . h...",CHIEF COMPLAINT\n\nHigh blood pressure check.\...
4,virtassist,D2N132,"[doctor] hello mrs. lee , i see you're here fo...",CC:\n\nBack pain.\n\nHPI:\n\nMs. Lee is a 40-y...


# Fine-tuning using Aci-Bench train set and MedSynth

In [15]:
# log into huggingface to save the fine-tuned model:

from huggingface_hub import notebook_login

notebook_login()

## Creating the instruction-tuning dataset

In [ ]:
import pandas as pd
aci_train= pd.read_csv("/content/TaskC-TrainingSet.csv")
aci_train.head()

,dataset,encounter_id,dialogue,note
0,virtassist,D2N001,"[doctor] hi , martha . how are you ?\n[patient...",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...
1,virtassist,D2N002,"[doctor] hi , andrew , how are you ?\n[patient...",CHIEF COMPLAINT\n\nJoint pain.\n\nHISTORY OF P...
2,virtassist,D2N003,"[doctor] hi , john . how are you ?\n[patient] ...",CHIEF COMPLAINT\n\nBack pain.\n\nHISTORY OF PR...
3,virtassist,D2N004,"[doctor] hi , james , how are you ?\n[patient]...",CHIEF COMPLAINT\n\nBack pain.\n\nHISTORY OF PR...
4,virtassist,D2N005,"[doctor] hey , ms. hill . nice to see you .\n[...",CC:\n\nRight middle finger pain.\n\nHPI:\n\nMs...


In [ ]:
from datasets import load_dataset

medsynth = load_dataset("Ahmad0067/MedSynth")
medsynth

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


MedSynth_huggingface_final.csv:   0%|          | 0.00/78.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10240 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: [' Note', 'Dialogue', 'ICD10', 'ICD10_desc'],
        num_rows: 10240
    })
})

In [ ]:
# combining them into one training set:

aci_target_cols = aci_train[["dialogue", "note"]].rename(columns={"dialogue": "Dialogue", "note": "Note"})

medsynth_target_cols= medsynth["train"].to_pandas()
# Clean column names
medsynth_target_cols.columns = medsynth_target_cols.columns.str.strip()

# Select and standardize
medsynth_target_cols = medsynth_target_cols[["Dialogue", "Note"]]

data_combined = pd.concat([aci_target_cols, medsynth_target_cols], ignore_index=True)
len(data_combined)
data_combined.to_csv("/content/data_combined.csv")

In [ ]:


DATASETS_PATHS= ["/content/data_combined.csv"]
llama3_datasets = []
for dataset_path in DATASETS_PATHS:

        dataset_name = dataset_path.split(os.sep)[-1].split(".")[0]

        llama3_dataset = process_dataset(dataset_path, "llama3")
        llama3_datasets.append(llama3_dataset)


llama3_dataset = pd.concat(llama3_datasets, ignore_index=True)
llama3_dataset= llama3_dataset.rename(columns={"prompt": "text"})
llama3_dataset = create_dataset_hf(llama3_dataset)

#push the dataset to hub
llama3_dataset.push_to_hub("test_aci_medsynth")


<|begin_of_text|><|start_header_id|>system<|end_header_id|> 

 {{ You are an assistant for medical professionals, specializing in summarizing their conversations with patients. Your role is to accurately and comprehensively summarize these conversations in the SOAP (Subjective, Objective, Assessment, Plan) format. Ensure that each summary is thorough and precise, reflecting all relevant details from the conversation to provide a reliable medical record. }}<|eot_id|><|start_header_id|>user<|end_header_id|>

  {{ This is the conversation: [doctor] hi , martha . how are you ?
[patient] i'm doing okay . how are you ?
[doctor] i'm doing okay . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you , okay ?
[patient] okay .
[doctor] martha is a 50-year-old female with a past medical history significant for congestive heart failure , depression and hypertension who presents for her annual exam . so , martha , it's been a year since i've seen you . how are you d

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Ahmad0067/test_aci_medsynth/commit/cc94100d40ad60241d0299dc61bba47d0084591a', commit_message='Upload dataset', commit_description='', oid='cc94100d40ad60241d0299dc61bba47d0084591a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Ahmad0067/test_aci_medsynth', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Ahmad0067/test_aci_medsynth'), pr_revision=None, pr_num=None)

## Instruction Fine-tuning

In [ ]:
os.environ["WANDB_DISABLED"] = "true"

dataset_hf_path= "Ahmad0067/test_aci_medsynth"
tuner = ModelTuner(TRAINING_DATA_PATH_HF= dataset_hf_path,
                                  FINE_TUNED_MODEL_NAME= f"test_aci_medsynth",
                                  tuning_config= tuning_config, base_model= base_model_path)

tuner.model_train_and_save()

==((====))==  Unsloth 2025.7.5: Fast Llama patching. Transformers: 4.53.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,307 | Num Epochs = 1 | Total steps = 0
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.625100


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


README.md:   0%|          | 0.00/594 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Saved model to https://huggingface.co/test_aci_medsynth


## Testing the fine-tuned model:

In [ ]:
# Loading the model and the tokenizer for inference
fine_tuned_model, fine_tuned_tokenizer=  FastLanguageModel.from_pretrained(model_name = "Ahmad0067/test_aci_medsynth",
                                                                        max_seq_length = tuning_config.get("model_config").get("max_seq_length"),
                                                                        dtype = tuning_config.get("model_config").get("dtype"),
                                                                        load_in_4bit = tuning_config.get("model_config").get("load_in_4bit"),)

FastLanguageModel.for_inference(fine_tuned_model)

==((====))==  Unsloth 2025.7.5: Fast Llama patching. Transformers: 4.53.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Unsloth 2025.7.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
fine_tuned_model_responses= get_model_responses(model= fine_tuned_model, tokenizer= fine_tuned_tokenizer,
                                                summarizer_system_promt= summarizer_system_prompt,
                                                test_dataset= aci_test, generation_config= model_evaluator_generation_config)

processing idx: 0
processing idx: 1
processing idx: 2
processing idx: 3
processing idx: 4
processing idx: 5
processing idx: 6
processing idx: 7
processing idx: 8
processing idx: 9
processing idx: 10
processing idx: 11
processing idx: 12
processing idx: 13
processing idx: 14
processing idx: 15
processing idx: 16
processing idx: 17
processing idx: 18
processing idx: 19
processing idx: 20
processing idx: 21
processing idx: 22
processing idx: 23
processing idx: 24
processing idx: 25
processing idx: 26
processing idx: 27
processing idx: 28
processing idx: 29
processing idx: 30
processing idx: 31
processing idx: 32
processing idx: 33
processing idx: 34
processing idx: 35
processing idx: 36
processing idx: 37
processing idx: 38
processing idx: 39


In [ ]:
fine_tuned_model_metrics= get_automatic_eval_scores(fine_tuned_model_responses, aci_test)

fine_tuned_model_metrics

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


{'BLEU': 0.09984726317290137,
 'ROUGE-1': np.float64(0.49140216820503285),
 'ROUGE-2': np.float64(0.1939046343140281),
 'ROUGE-L': np.float64(0.2678378354575769),
 'ROUGE-LSum': np.float64(0.4496342076744206),
 'METEOR': np.float64(0.28880670124826635)}